# IMPORTAR PAQUETES

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import janitor
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import RFE
from xgboost import XGBRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


# EMPAQUETAR TODO EL PROCESO DE FUNCIONES

## Calidad de datos

In [ ]:
def calidad_datos(x, num=None):

    # Modificar los tipos
    temp = x.astype({'month': 'O', 'wday': 'O'})

    # Imputar los nulos
    temp.loc[x['event_name_1'].isna(), 'event_name_1'] = 'Sin_evento'

    def imputar_moda(registros):
        # Calcular la moda del precio en ese grupo semana-producto
        moda_serie = registros.sell_price.mode()
        if not moda_serie.empty:
            moda = moda_serie.iloc[0]
        else:
            # Si no hay moda en esa semana, calcular la moda a nivel de producto
            item_id = registros.item_id.iloc[0]
            if num is not None:
                moda_producto = num[num.item_id == item_id].sell_price.mode()
                if not moda_producto.empty:
                    moda = moda_producto.iloc[0]
                else:
                    moda = np.nan
            else:
                moda = np.nan

        registros.loc[registros.sell_price.isna(), 'sell_price'] = moda
        return registros

    temp = temp.groupby('item_id', group_keys=False).apply(imputar_moda)
    return temp

## Creación de variables

In [ ]:
def crear_variables(x):

    # DEMANDA INTERMITENTE

    def rotura_stock(ventas, n = 5):
        cero_ventas = pd.Series(np.where(ventas == 0, 1, 0))
        num_ceros = cero_ventas.rolling(n).sum()
        rotura_stock = np.where(num_ceros == n, 1, 0)
        return (rotura_stock)

    x = x.sort_values(by = ['store_id','item_id','date'])
    x['rotura_stock_3'] = x.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 3)).values
    x['rotura_stock_7'] = x.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 7)).values
    x['rotura_stock_15'] = x.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 15)).values

    # LAGS

    def crear_lags(df, variable, num_lags = 7):

        # Crear el objeto dataframe
        lags = pd.DataFrame()

        # Crea todos los lags 
        for cada in range(1, num_lags + 1):
            lags[variable + '_lag_' + str(cada)] = df[variable].shift(cada)

        # Devuelve el dataframe con los lags    
        return(lags)

    lags_sell_price_df = (x.groupby(['store_id', 'item_id'])
                        .apply(lambda x: crear_lags(x = x, variable = 'sell_price', num_lags= 7))
                        .reset_index()
                        .set_index('date'))

    lags_rotura_stock_3_df = (x.groupby(['store_id', 'item_id'])
                            .apply(lambda x: crear_lags(x = x, variable = 'rotura_stock_3', num_lags= 1))
                            .reset_index()
                            .set_index('date'))

    lags_rotura_stock_7_df = (x.groupby(['store_id', 'item_id'])
                                .apply(lambda x : crear_lags(x = x, variable= 'rotura_stock_7', num_lags= 1))
                                .reset_index()
                                .set_index('date'))

    lags_rotura_stock_15_df = (x.groupby(['store_id', 'item_id'])
                                .apply(lambda x : crear_lags(x = x, variable= 'rotura_stock_15', num_lags= 1))
                                .reset_index()
                                .set_index('date'))

    lags_ventas_df = (x.groupby(['store_id', 'item_id'])
                        .apply(lambda x : crear_lags(x = x, variable='ventas', num_lags= 15))
                        .reset_index()
                        .set_index('date'))


    # Ventanas MÓVILES

    def min_movil(df, variable, num_periodos= 7):

        minm = pd.DataFrame()

        for cada in range(2, num_periodos+1):
            minm[variable + '_minm_' + str(cada)] = df[variable].shift(1).rolling(cada).min()

        return(minm)


    def media_movil(df, variable, num_periodos= 7):

        mm = pd.DataFrame()

        for cada in range(2, num_periodos+1):
            mm[variable + '_mm_' + str(cada)] = df[variable].shift(1).rolling(cada).mean()

        return(mm)


    def max_movil(df, variable, num_periodos= 7):

        maxm = pd.DataFrame()

        for cada in range(2, num_periodos+1):
            maxm[variable + '_maxm_' + str(cada)] = df[variable].shift(1).rolling(cada).max()

        return(maxm)


    min_movil_df = (x.groupby(['store_id', 'item_id'])
                    .apply(lambda x : min_movil(df = x, variable= 'ventas', num_periodos= 15))
                    .reset_index()
                    .set_index('date'))

    media_movil_df = (x.groupby(['store_id', 'item_id'])
                    .apply(lambda x : media_movil(df = x, variable= 'ventas', num_periodos= 15))
                    .reset_index()
                    .set_index('date'))

    max_movil_df = (x.groupby(['store_id', 'item_id'])
                    .apply(lambda x : max_movil(df = x, variable= 'ventas', num_periodos= 15))
                    .reset_index()
                    .set_index('date'))

    # UNIR DATAFRAMES CREADOS

    x_unido = pd.concat([x,
             lags_sell_price_df,
             lags_rotura_stock_3_df,
             lags_rotura_stock_7_df,
             lags_rotura_stock_15_df,
             lags_ventas_df,
             min_movil_df,
             media_movil_df,
             max_movil_df], axis = 1 )

    # Eliminar columnas duplicadas
    x_unido = x_unido.loc[:,~x_unido.columns.duplicated()]

    x_unido.dropna(inplace=True)

    x_unido.drop(columns= ['sell_price', 'rotura_stock_3', 'rotura_stock_7', 'rotura_stock_15'], 
                 inplace=True)

    # Crear una sola variable para producto

    x_unido = x_unido.insert(loc=0, column='Producto', value=x_unido.store_id + ' ' + x_unido.item_id)

    x_unido = x_unido.drop(columns= ['store_id', 'item_id'])

    return (x_unido)

## Transformación de Variables

In [ ]:
def transformar_variables(x, y=None, modo='entrenamiento')
    '''
    He modificado esta función para que sirva tanto para entrenamiento como ejecución:

    * Incluyendo el parámetro modo que por defecto es entrenamiento
    * Haciendo que el parámetro 'y' sea opcional, ya que en ejecución no se usa

    Cuando se usa en modo entrenamiento aplica el método fit_transform y guarda el objeto.

    Mientras que cuando esta en modo ejecución carga los objetos y aplica sobre el método transform. 

    '''
    x.reset_index(inplace=True)

    #GESTIÓN DE ENCODERS
    nombre_ohe = 'ohe_retail.pkl'
    nombre_te = 'te_retail.pkl'
    ruta_ohe = '../05_modelos/' + nombre_ohe
    ruta_te = '../05_modelos/' + nombre_te

    #ONE HOT ENCODING
    var_ohe = ['event_name_1']
    if modo == 'entrenamiento':
        #Si esta en entrenamiento aplica fit_transform y guarda el encoder
        ohe = OneHotEncoder(sparce_output = False, handle_unkown = 'ignore')
        ohe_x = ohe.fit_transform(x[var_ohe])
        ohe_x = pd.DataFrame(ohe_x, columns = ohe_x.get_feature_names_out())
        with open(ruta_ohe, mode='wb') as file:
            pickle.dump(ohe, file)
    else:
        #Si está en ejecución recuperar el guardado y solo aplica tranform
        with open(ruta_ohe, mode='rb') as file:
            ohe = pickle.load(file)
        ohe_x = ohe.transform(x[var_ohe])
        ohe_x = pd.DataFrame(ohe_x, columns = ohe_x.get_feature_names_out())

    #TARGET ENCODING
    var_te = ['month', 'wday', 'weekday']
    if modo == 'entrenamiento':
        #ASEGURAR QUE 'Y' TIENE LOS MISMO REGISTROS QUE 'X'
        y.reset_index(inplace=True, drop = True)
        y = y.loc[y.index.isin(x.index)]
        #Si esta en entrenamiento aplica fit_transform y guarda el encoder
        te = TargetEnconder(min_leaf = 100, return_df = False)
        te_x = te.fit_transform(x[te_x], y = y)
        nombres_te = [ variable + '_te' for variable in var_te]
        te_x = pd.DataFrame(te_x, columns = nombres_te)
        with open(ruta_te, mode='wb') as file:
            pickle.dump(te, file)
    else:
        #Si está en ejecución recupera el guardado y solo aplica transform
        with open(ruta_te, mode='rb') as file:
            te = pickle.load(file)
        te_x = te.transform(x[te_x])
        nombres_te = [variable + '_te' for variable in var_te]
        te_x = pd.DataFrame(te_x, columns = nombres_te)

    #INTEGRAR, LIMPIAR Y DEVOLVER EL DATAFRAME
    #Eliminar las originales ya transformadas
    x = x.drop(['event_name_1', 'month', 'wday', 'weekday'])
    #Incorporar los otros dataframes
    x = pd.concat([x, ohe_x, te_x], axis=1).set_index('date')

    return(x)

## Preselección de Variables

In [ ]:
def preseleccion_variables(x, y):
    '''
    Solo se usa en entrenamiento
    '''

    #ELIMINAR LA COLUMNA PRODUCTO E INDICE
    x.reset_index(drop = True, inplace = True)
    x.drop(columns = 'producto', inplace = True)

    #Asegurar que y tiene los mismos registros que x
    y = y.loc[y.index.isin(x.index)]

    mutual_selector = mutual_info_regression(x, y)
    posicion_variable_limite = 75
    ranking_mi = pd.DataFrame(mutual_selector, index = x.columns).reset_index()
    ranking_mi.columns = ['variable', 'importancia_mi']
    ranking_mi = ranking_mi.sort_values( by = 'importancia_mi', ascending = False)
    ranking_mi['ranking_mi'] = np.arange(0, ranking_mi.shape[0])
    entran_mi = ranking_mi.iloc[0:posicion_variable_limite].variable
    x_mi = x[entran_mi].copy()

    return (x_mi)

## Modelización

### Entrenamiento

In [ ]:
def modelizar (x_producto, y):

    '''
    Esta función es la que hace la modelización individual.

    Recibe los datos de las x y las y de un producto.

    Encuentra los parámetros óptimos para ese producto.
    
    Devuele el mejor modelo.
    '''

    #Excluye el producto como variable de modelización
    var_modelizar = x_producto.columns.to_lits()[2:]

    #Define la validación cruzada
    time_cv = TimeSeriesSplit(5, test_size=8)

    #Define la parrilla de algoritmos
    pipe = Pipeline([('algoritmo', HistGradientBoostingRegressor())])

    grid = [
        {'algoritmo' : [HistGradientBoostingRegressor()]
         # 'algoritmo__learning_rate': [0.01, 0.025, 0.05, 0.1],
         # 'algoritmo__max_iter': [300, 500, 800],
         # 'algoritmo__max_depth': [3, 5, 10]
         # 'algoritmo__scoring' : ['neg_mean_absolute_error]
         # 'algoritmo__12_regularization' : [0, 0.25, 0,5, 0.75, 1]
        }
    ]

    random_search = RandomizedSearchCV(estimator = pipe,
                                   param_distributions = grid,
                                   n_iter = 1,
                                   cv = time_cv,
                                   scoring = 'neg_mean_absolute_error',
                                   verbose = 0,
                                   n_jobs = -1)
    
    modelo = random_search.fit(x_producto[var_modelizar], y)

    #Reentrena el mejor sobre todos los datos 
    modelo_final = modelo.best_estimator_.fit(x_producto[var_modelizar], y)

    #Devuelve como salida el módelo final
    return (modelo_final)

In [ ]:
def lanzar_entrenamiento (df):

    '''
    Esta función va recorriendo todos los productos y llamando a modelizar() para crear una lista total con todos

    Recibe el dataframe de las x ya limpio y segmentado por producto, y tambien por target.

    No devuelve nada si no que guarda en disco el objeto ya entrenado con todos los modelos.
    '''

    lista_productos = list(df.producto.unique())

    lista_modelos = []

    for cada in lista_productos:

        #Renombrar con claridad
        producto = cada
        target = 'ventas'

        x = df.loc[df.producto == producto].copy().drop(columns=target).copy()
        y = df.loc[df.producto == producto, 'ventas'].copy()

        #Llama a la función de modelizar
        modelo = modelizar(x, y)

        #Añade el modelo final a la lista productos
        lista_modelos.append((producto, modelo))

    #Guarda la lista de modelos entrenados
    nombre_modelos = 'lista_modelos_retail.pkl'
    ruta_modelos = '../05_modelos/' + nombre_modelos
    with open(ruta_modelos, mode='wb') as file:
        pickle.dump(lista_modelos, file)

### Ejecución

In [ ]:
def lanzar_ejecucion (df):

    '''
    Esta función hace el forecast para cada producto, pero solo de un día

    Recíbe el nuevo dataset a predecir.

    Que tiene que tener la estructura del fichero DatosParaProduccion.csv de la carpeta Validación

    Va recorriendo cada producto, cargando su módelo correspondiente, seleccionando sus datos, y haciendo sus predicciones

    Devuelve la predicción para todos los productos pero SOLO PARA EL DÍA QUE TOCA.
    '''

    #CARGA LOS MODELOS
    nombre_modelos = 'lista_modelos_retail.pkl'
    ruta_modelos = '../05_modelos/' + nombre_modelos
    with open(ruta_modelos, mode='rb') as file:
        lista_modelos = pickle.load(file)

    predicciones_df = pd.DataFrame(columns=['date', 'producto', 'ventas', 'predicccion'])

    for cada in range(0,len(lista_modelos)):

        # Prepara los datos necesarios
        producto = lista_modelos[cada][0]
        modelo = lista_modelos[cada][1]
        variables = modelo[0].feature_names_in_
        target = 'ventas'

        x = df.loc[df.producto == producto].drop(columns=target).copy()
        y = df.loc[df.producto == producto, 'ventas'].copy()

        date = df.reset_index().copy()
        date = date.loc[date.producto == producto, 'date'].values

        #Transformacion de variables
        x = transformar_variables(x, modo = 'ejecucion')

        #Selección de variables
        x = x[variables]

        #Cálculo de predicciones
        predicciones = pd.DataFrame(data= {'date' : date,
                                           'producto' : producto,
                                           'ventas' : y,
                                           'prediccion' : modelo.predict(x)})

        predicciones['prediccion'] = predicciones.prediccion.astype('int')

        predicciones_df = pd.concat([predicciones_df, predicciones])

    predicciones_df = predicciones_df.loc[predicciones_df.index == predicciones_df.index.min()]
    return (predicciones_df)

In [ ]:
def forecast_recursivo (x):

    '''
    Esta función es la que aplica el forecast recursivo para predecir 8 días.

    Recibe el nuevo dataset a predecir.

    Que tiene que tener la estructura del fichero DatosParaProduccion.csv de la carpeta de Validación.

    Ya que para aplicar al recursividad:

    * Va a predecir el primer día para el cual tenga toda la información (es decir 15 días desde el día mas antiguo)
    * Al finalizar graba la predicción de ventas en el registro a predecir y elimina los registros del día mas antiguo
    * Por tanto en la siguiente iteracion va a predecir el siguiente día.

    Por ejemplo:

    Si el día más antiguo del dataset es el 09/12/2015 entonces el primer día que puede predecir

    (y del cual ya no tenemos dato) es el 24/12/2015.

    Cuando predice el dato del 24 para cada producto lo sobreescribe como sus ventas

    y elimina todos los registros del día 09.

    Entonces el día mas antiguo pasa a ser el día 10 y por tanto el día a predecir es el 25.

    Y así hasta que finaliza 8 ciclos para predecir la semana que queremos.
    '''

    for cada in range(0, 8):
        paso1_df = calidad_datos(x)
        paso2_df = crear_variables(paso1_df)

        #Calcula la predicción
        f = lanzar_ejecucion(paso2_df)
        f['store_id'] = f.producto.str[:4]
        f['item_id'] = f.producto.str[5:]

        #Actualiza el dato de ventas con la predicción
        x.loc[(x.index.isin(f.date)) & (x.store_id.isin(f.store_id)) & (x.item_id.isin(f.item_id)), 'ventas'] = f.producto

        #Elimina el día más antiguo de x
        x = x.loc[x.index != x.index.min()]

    return (x)
